In [1]:
# variation of skill with season
# variation of skill with initial WR
# variation of skill with ENSO

import copy
import glob
import os
import sys
from datetime import datetime, timedelta
import string

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# import visualkeras
# import tensorflow as tf
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight

sys.path.append("/glade/u/home/jhayron/WR_Predictability/3_MLModels/")
from sklearn import datasets, ensemble
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# persistence performance

In [2]:
wr_series = pd.read_csv('/glade/work/jhayron/Data4Predictability/WR_Series_20231120.csv',\
                index_col=0,names=['week0'],skiprows=1,parse_dates=True)
for wk in range(2,10):
    series_temp = copy.deepcopy(wr_series["week0"])
    series_temp.index = series_temp.index - timedelta(weeks = wk-1)
    series_temp.name = f'week{wk-1}'
    if wk==2:
        df_shifts = pd.concat([pd.DataFrame(wr_series["week0"]),pd.DataFrame(series_temp)],axis=1)  
    else:
        df_shifts = pd.concat([df_shifts,pd.DataFrame(series_temp)],axis=1)

In [3]:
acc_persistence = []

for wk in range(1, 9):
    df_temp = df_shifts[[f"week{0}", f"week{wk}"]].dropna()  # ['2016':'2020']
    acc_persistence.append(
        f1_score(df_temp[f"week{wk}"], df_temp[f"week{0}"], average="micro")
    )

In [4]:
acc_persistence = np.array(acc_persistence)

In [5]:
np.save('data_figures_2_5/acc_persistence.npy',acc_persistence)

# performance xgb

In [6]:
path_results = "results/"
list_folders = np.sort(glob.glob('/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/figures_detrended_20240317/*/'))
list_vars = [list_folders[i].split('/')[-2] for i in range(len(list_folders))]

for ivar,var in enumerate(list_vars):
    print(ivar,var)

indices_atmosphere = np.array([8, 21, 22, 23])
indices_land = np.array([9, 13, 14, 15, 16, 17, 18, 19, 20])
# indices_land = np.array([9,12,13,14,15,16,18,19])
indices_ocean = np.array([0, 1, 2, 3, 4, 5, 6, 7, 10, 12])
# indices_ocean = np.array([0, 1, 2, 3, 4, 5, 6, 7, 10, 12])

0 IC_SODA
1 IT_SODA
2 MLD_SODA
3 OHC100_SODA
4 OHC200_SODA
5 OHC300_SODA
6 OHC50_SODA
7 OHC700_SODA
8 OLR_ERA5
9 SD_ERA5
10 SSH_SODA
11 SST_OISSTv2
12 SST_SODA
13 STL_1m_ERA5
14 STL_28cm_ERA5
15 STL_7cm_ERA5
16 STL_full_ERA5
17 SWVL_1m_ERA5
18 SWVL_28cm_ERA5
19 SWVL_7cm_ERA5
20 SWVL_full_ERA5
21 U10_ERA5
22 U200_ERA5
23 Z500_ERA5
24 Z500_ERA5_Region


In [7]:
# Define weeks
weeks = np.arange(1, 9)

In [8]:
np.random.seed(42)

## Evaluate atmospheric variables

In [9]:
def compute_accuracy_and_range(results_temp):
    accuracy_iterations = []
    for _ in range(1000):  # Adjust the number of iterations as needed
        sample_indices = np.random.choice(results_temp.index, size=int(len(results_temp)), replace=True)
        sampled_data = results_temp.loc[sample_indices]
        accuracy_iterations.append(f1_score(sampled_data['y_true'], sampled_data['y_predicted'],average='micro'))
    # accuracy_mean = np.mean(accuracy_iterations)
    accuracy_mean = f1_score(results_temp['y_true'],results_temp['y_predicted'],average='micro')
    accuracy_95th = np.percentile(accuracy_iterations,95)
    accuracy_5th = np.percentile(accuracy_iterations,5)
    return accuracy_mean, accuracy_95th, accuracy_5th
    
def process_results(indices, list_vars, path_results):
    all_accs = []
    all_uppers = []
    all_lowers = []
    
    weeks = np.arange(1, 9)
    
    for ivar in range(len(indices)):
        name_temp = list_vars[indices[ivar]]
        acc_temp = []
        acc_upper_temp = []
        acc_lower_temp = []
        for week in weeks:
            results_temp = pd.read_csv(f'{path_results}results_{name_temp}_week{week}.csv', index_col=0, parse_dates=True)
            acc_mean, acc_upper, acc_lower = compute_accuracy_and_range(results_temp)
            acc_temp.append(acc_mean)
            acc_upper_temp.append(acc_upper)
            acc_lower_temp.append(acc_lower)
        all_accs.append(acc_temp)
        all_uppers.append(acc_upper_temp)
        all_lowers.append(acc_lower_temp)
    
    if 'SSH_SODA' in np.array(list_vars)[indices]:
        component = 'ocn'
    if 'Z500_ERA5' in np.array(list_vars)[indices]:
        component = 'atm'
    if 'SWVL_full_ERA5' in np.array(list_vars)[indices]:
        component = 'lnd'
        
    ## the joint model
    name_temp = 'ALL'
    acc_temp = []
    acc_upper_temp = []
    acc_lower_temp = []
    for week in weeks:
        results_temp = pd.read_csv(f'{path_results}results_{component}_week{week}.csv', index_col=0, parse_dates=True)
        acc_mean, acc_upper, acc_lower = compute_accuracy_and_range(results_temp)
        acc_temp.append(acc_mean)
        acc_upper_temp.append(acc_upper)
        acc_lower_temp.append(acc_lower)
    all_accs.append(acc_temp)
    all_uppers.append(acc_upper_temp)
    all_lowers.append(acc_lower_temp)
    
    ind_max_acc = np.argmax(np.array(all_accs), axis=0)
    
    best_model = []
    for ind in ind_max_acc:
        if ind<len(indices):
            best_model.append('\n'.join(list_vars[indices[ind]].split('_')[:-1]))
        else:
            best_model.append('All')
    print(best_model)
    # best_model = ['\n'.join(list_vars[indices[ind]].split('_')[:-1]) for ind in ind_max_acc]
    
    max_acc_upper = [np.array(all_uppers)[ind_max_acc[i], i] for i in range(len(ind_max_acc))]
    max_acc_lower = [np.array(all_lowers)[ind_max_acc[i], i] for i in range(len(ind_max_acc))]
    
    best_model_all = best_model
    best_acc_all = np.max(np.array(all_accs), axis=0)
    best_acc_upper_all = np.array(max_acc_upper)
    best_acc_lower_all = np.array(max_acc_lower)
    mean_acc_all = np.mean(np.array(all_accs), axis=0)
    diff_acc_all = best_acc_all - mean_acc_all
    
    return all_accs, all_uppers, all_lowers, best_model_all, best_acc_all, \
        best_acc_upper_all, best_acc_lower_all, mean_acc_all, diff_acc_all

In [10]:
# Call the function with necessary arguments for ocean
dic_ocean = {}
dic_ocean['all_accs_ocean'], dic_ocean['all_uppers_ocean'], dic_ocean['all_lowers_ocean'], \
    dic_ocean['best_model_ocean'], dic_ocean['best_acc_ocean'], dic_ocean['best_acc_upper_ocean'], \
    dic_ocean['best_acc_lower_ocean'], dic_ocean['mean_acc_ocean'], \
    dic_ocean['diff_acc_ocean'] = process_results(indices_ocean, list_vars, path_results)

# Call the function with necessary arguments for land
dic_land = {}
dic_land['all_accs_land'], dic_land['all_uppers_land'], dic_land['all_lowers_land'], \
    dic_land['best_model_land'], dic_land['best_acc_land'], dic_land['best_acc_upper_land'], \
    dic_land['best_acc_lower_land'], dic_land['mean_acc_land'], \
    dic_land['diff_acc_land'] = process_results(indices_land, list_vars, path_results)

# Call the function with necessary arguments for atmosphere
dic_atmosphere = {}
dic_atmosphere['all_accs_atmosphere'], dic_atmosphere['all_uppers_atmosphere'], dic_atmosphere['all_lowers_atmosphere'], \
    dic_atmosphere['best_model_atmosphere'], dic_atmosphere['best_acc_atmosphere'], dic_atmosphere['best_acc_upper_atmosphere'], \
    dic_atmosphere['best_acc_lower_atmosphere'], dic_atmosphere['mean_acc_atmosphere'], \
    dic_atmosphere['diff_acc_atmosphere'] = process_results(indices_atmosphere, list_vars, path_results)

['All', 'OHC700', 'OHC700', 'OHC700', 'OHC700', 'OHC100', 'OHC200', 'OHC300']
['STL\n28cm', 'STL\n7cm', 'SWVL\n7cm', 'SWVL\nfull', 'STL\nfull', 'SWVL\n1m', 'STL\nfull', 'STL\nfull']
['Z500', 'All', 'All', 'Z500', 'All', 'All', 'All', 'U200']


In [11]:
np.save('data_figures_2_5/dic_ocean.npy', dic_ocean)
np.save('data_figures_2_5/dic_land.npy', dic_land)
np.save('data_figures_2_5/dic_atmosphere.npy', dic_atmosphere)

# Check for seasons

In [12]:
indices_seasons = {'Winter':[12,1,2],
                   'Spring':[3,4,5],
                   'Summer':[6,7,8],
                   'Fall':[9,10,11]}

acc_persistence_seasons = {}
for season in indices_seasons.keys():
    acc_persistence_temp = []
    for wk in range(1,9):
        df_temp = df_shifts[[f'week{0}',f'week{wk}']].dropna()
        df_temp = df_temp[(df_temp.index.month==indices_seasons[season][0])|\
                                    (df_temp.index.month==indices_seasons[season][1])|\
                                    (df_temp.index.month==indices_seasons[season][2])]
        acc_persistence_temp.append(f1_score(df_temp[f'week{wk}'],df_temp[f'week{0}'],average='micro'))
    acc_persistence_seasons[season] = acc_persistence_temp

In [13]:
np.save('data_figures_2_5/acc_persistence_seasons.npy',acc_persistence_seasons)

In [14]:
def process_results_with_seasons(indices, list_vars, path_results, indices_seasons, season):
    all_accs = []
    all_uppers = []
    all_lowers = []
    weeks = np.arange(1, 9)
    
    for ivar in range(len(indices)):
        name_temp = list_vars[indices[ivar]]
        acc_temp = []
        acc_upper_temp = []
        acc_lower_temp = []
        
        for week in weeks:
            results_temp = pd.read_csv(f'{path_results}results_{name_temp}_week{week}.csv', index_col=0, parse_dates=True)
            results_temp = results_temp[(results_temp.index.month==indices_seasons[season][0])|\
                            (results_temp.index.month==indices_seasons[season][1])|\
                            (results_temp.index.month==indices_seasons[season][2])]
            
            acc_mean, acc_upper, acc_lower = compute_accuracy_and_range(results_temp)
            acc_temp.append(acc_mean)
            acc_upper_temp.append(acc_upper)
            acc_lower_temp.append(acc_lower)
        
        all_accs.append(acc_temp)
        all_uppers.append(acc_upper_temp)
        all_lowers.append(acc_lower_temp)
        
    if 'SSH_SODA' in np.array(list_vars)[indices]:
        component = 'ocn'
    if 'Z500_ERA5' in np.array(list_vars)[indices]:
        component = 'atm'
    if 'SWVL_full_ERA5' in np.array(list_vars)[indices]:
        component = 'lnd'
        
    ## the joint model
    name_temp = 'ALL'
    acc_temp = []
    acc_upper_temp = []
    acc_lower_temp = []
    for week in weeks:
        results_temp = pd.read_csv(f'{path_results}results_{component}_week{week}.csv', index_col=0, parse_dates=True)
        results_temp = results_temp[(results_temp.index.month==indices_seasons[season][0])|\
                        (results_temp.index.month==indices_seasons[season][1])|\
                        (results_temp.index.month==indices_seasons[season][2])]
        acc_mean, acc_upper, acc_lower = compute_accuracy_and_range(results_temp)
        acc_temp.append(acc_mean)
        acc_upper_temp.append(acc_upper)
        acc_lower_temp.append(acc_lower)
    all_accs.append(acc_temp)
    all_uppers.append(acc_upper_temp)
    all_lowers.append(acc_lower_temp)
    
    ind_max_acc = np.argmax(np.array(all_accs), axis=0)
    
    best_model = []
    for ind in ind_max_acc:
        if ind<len(indices):
            best_model.append('\n'.join(list_vars[indices[ind]].split('_')[:-1]))
        else:
            best_model.append('All')

    max_acc_upper = [np.array(all_uppers)[ind_max_acc[i], i] for i in range(len(ind_max_acc))]
    max_acc_lower = [np.array(all_lowers)[ind_max_acc[i], i] for i in range(len(ind_max_acc))]

    best_model_all = best_model
    best_acc_all = np.max(np.array(all_accs), axis=0)
    best_acc_upper_all = np.array(max_acc_upper)
    best_acc_lower_all = np.array(max_acc_lower)
    mean_acc_all = np.mean(np.array(all_accs), axis=0)
    diff_acc_all = best_acc_all - mean_acc_all
    
    return all_accs, all_uppers, all_lowers, best_model_all, best_acc_all, \
        best_acc_upper_all, best_acc_lower_all, mean_acc_all, diff_acc_all

In [15]:
# Call the function with necessary arguments for each season
for iseason, season in enumerate(indices_seasons.keys()):
    best_accs = {}
    best_acc_uppers = {}
    best_acc_lowers = {}
    best_names = {}
    component_names = ['atmosphere','ocean','land']
    for indices_component, component in zip([indices_atmosphere, indices_ocean, indices_land], component_names):
        all_accs, all_uppers, all_lowers, best_model, best_acc, \
            best_acc_upper, best_acc_lower, mean_acc, diff_acc = \
                process_results_with_seasons(indices_component, list_vars, path_results, indices_seasons, season)
        best_accs[component] = best_acc
        best_acc_uppers[component] = best_acc_upper
        best_acc_lowers[component] = best_acc_lower
        best_names[component] = best_model
        print(best_names[component])

    dic_season = {}
    dic_season['best_accs'] = best_accs
    dic_season['best_acc_uppers'] = best_acc_uppers
    dic_season['best_acc_lowers'] = best_acc_lowers
    dic_season['best_names'] = best_names

    np.save(f'data_figures_2_5/dic_{season}.npy',dic_season)

['All', 'All', 'Z500', 'Z500', 'U10', 'U10', 'U200', 'All']
['OHC700', 'OHC700', 'OHC200', 'OHC700', 'OHC50', 'MLD', 'IC', 'OHC300']
['SD', 'STL\n7cm', 'SWVL\n7cm', 'STL\n28cm', 'STL\nfull', 'SWVL\n7cm', 'SD', 'STL\n1m']
['Z500', 'U200', 'All', 'Z500', 'Z500', 'All', 'All', 'U200']
['OHC700', 'SST', 'OHC300', 'OHC700', 'OHC100', 'OHC100', 'OHC700', 'SSH']
['STL\n28cm', 'STL\n7cm', 'SWVL\n1m', 'SWVL\nfull', 'SWVL\nfull', 'SWVL\n1m', 'STL\n7cm', 'STL\nfull']
['Z500', 'U200', 'Z500', 'Z500', 'OLR', 'Z500', 'U10', 'OLR']
['OHC300', 'OHC300', 'OHC700', 'OHC300', 'OHC200', 'OHC300', 'OHC300', 'OHC200']
['All', 'STL\n7cm', 'STL\n28cm', 'SWVL\nfull', 'SWVL\n1m', 'SWVL\n1m', 'SWVL\n1m', 'STL\nfull']
['Z500', 'All', 'All', 'U200', 'All', 'All', 'U10', 'Z500']
['MLD', 'OHC300', 'OHC300', 'OHC50', 'OHC50', 'OHC50', 'OHC200', 'OHC100']
['STL\n28cm', 'All', 'SD', 'STL\n28cm', 'STL\n7cm', 'STL\n7cm', 'STL\nfull', 'STL\nfull']


# Changes in predictability depending on WR

In [16]:
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score

In [17]:
names_wr = ["WR1: AR", "WR2: PT", "WR3: GH", "WR4: CUSH", "No WR"]

In [18]:
acc_persistence_wrs = {}
for iwr,wr in enumerate(names_wr):
    acc_persistence_temp = []
    for wk in range(1,9):
        df_temp = df_shifts[[f'week{0}',f'week{wk}']].dropna()
        df_temp = df_temp[(df_temp[f'week{wk}'] == iwr)|(df_temp[f'week{0}'] == iwr)]
        # df_temp = df_temp[(df_temp[f'week{wk}'] == iwr)]
        # df_temp = df_temp[(df_temp[f'week{0}'] == iwr)]
        acc_persistence_temp.append(accuracy_score(df_temp[f'week{wk}'],df_temp[f'week{0}']))
    acc_persistence_wrs[wr] = acc_persistence_temp

In [19]:
np.save('data_figures_2_5/acc_persistence_wrs.npy',acc_persistence_wrs)

In [20]:
def compute_accuracy_and_range(results_temp):
    accuracy_iterations = []
    for _ in range(1000):  # Adjust the number of iterations as needed
        sample_indices = np.random.choice(results_temp.index, size=int(len(results_temp)), replace=True)
        sampled_data = results_temp.loc[sample_indices]
        accuracy_iterations.append(accuracy_score(sampled_data['y_true'], sampled_data['y_predicted']))
    accuracy_mean = accuracy_score(results_temp['y_true'],results_temp['y_predicted'])
    accuracy_95th = np.percentile(accuracy_iterations,95)
    accuracy_5th = np.percentile(accuracy_iterations,5)

    return accuracy_mean, accuracy_95th, accuracy_5th

def process_results_with_wr(indices, list_vars, path_results, wr_temp):
    all_accs = []
    all_uppers = []
    all_lowers = []
    weeks = np.arange(1, 9)
    
    for ivar in range(len(indices)):
        name_temp = list_vars[indices[ivar]]
        acc_temp = []
        acc_upper_temp = []
        acc_lower_temp = []
        
        for week in weeks:
            results_temp = pd.read_csv(f'{path_results}results_{name_temp}_week{week}.csv', index_col=0, parse_dates=True)
            results_temp = results_temp[(results_temp['y_true']==wr_temp)|(results_temp['y_predicted']==wr_temp)]
            # results_temp = results_temp[(results_temp['y_true']==wr_temp)]
            # results_temp = results_temp[(results_temp['y_predicted']==wr_temp)]
            
            acc_mean, acc_upper, acc_lower = compute_accuracy_and_range(results_temp)
            acc_temp.append(acc_mean)
            acc_upper_temp.append(acc_upper)
            acc_lower_temp.append(acc_lower)
        
        all_accs.append(acc_temp)
        all_uppers.append(acc_upper_temp)
        all_lowers.append(acc_lower_temp)
        
    if 'SSH_SODA' in np.array(list_vars)[indices]:
        component = 'ocn'
    if 'Z500_ERA5' in np.array(list_vars)[indices]:
        component = 'atm'
    if 'SWVL_full_ERA5' in np.array(list_vars)[indices]:
        component = 'lnd'
        
    ## the joint model
    name_temp = 'ALL'
    acc_temp = []
    acc_upper_temp = []
    acc_lower_temp = []
    for week in weeks:
        results_temp = pd.read_csv(f'{path_results}results_{component}_week{week}.csv', index_col=0, parse_dates=True)
        results_temp = results_temp[(results_temp['y_true']==wr_temp)|(results_temp['y_predicted']==wr_temp)]
        acc_mean, acc_upper, acc_lower = compute_accuracy_and_range(results_temp)
        acc_temp.append(acc_mean)
        acc_upper_temp.append(acc_upper)
        acc_lower_temp.append(acc_lower)
    all_accs.append(acc_temp)
    all_uppers.append(acc_upper_temp)
    all_lowers.append(acc_lower_temp)
    
    ind_max_acc = np.argmax(np.array(all_accs), axis=0)
    
    best_model = []
    for ind in ind_max_acc:
        if ind<len(indices):
            best_model.append('\n'.join(list_vars[indices[ind]].split('_')[:-1]))
        else:
            best_model.append('All')

    max_acc_upper = [np.array(all_uppers)[ind_max_acc[i], i] for i in range(len(ind_max_acc))]
    max_acc_lower = [np.array(all_lowers)[ind_max_acc[i], i] for i in range(len(ind_max_acc))]

    best_model_all = best_model
    best_acc_all = np.max(np.array(all_accs), axis=0)
    best_acc_upper_all = np.array(max_acc_upper)
    best_acc_lower_all = np.array(max_acc_lower)
    mean_acc_all = np.mean(np.array(all_accs), axis=0)
    diff_acc_all = best_acc_all - mean_acc_all
    
    return all_accs, all_uppers, all_lowers, best_model_all, best_acc_all, \
        best_acc_upper_all, best_acc_lower_all, mean_acc_all, diff_acc_all

In [21]:
import warnings

warnings.filterwarnings("ignore")

In [22]:
# Call the function with necessary arguments for each season

for weather_regime,weather_regime_name in enumerate(names_wr):
    best_accs = {}
    best_acc_uppers = {}
    best_acc_lowers = {}
    best_names = {}
    component_names = ['atmosphere','ocean','land']
    for indices_component, component in zip([indices_atmosphere, indices_ocean, indices_land], component_names):
        all_accs, all_uppers, all_lowers, best_model, best_acc, \
            best_acc_upper, best_acc_lower, mean_acc, diff_acc = \
                process_results_with_wr(indices_component, list_vars, path_results, weather_regime)
        best_accs[component] = best_acc
        best_acc_uppers[component] = best_acc_upper
        best_acc_lowers[component] = best_acc_lower
        best_names[component] = best_model
        print(best_names[component])
    
    dic_wr = {}
    dic_wr['best_accs'] = best_accs
    dic_wr['best_acc_uppers'] = best_acc_uppers
    dic_wr['best_acc_lowers'] = best_acc_lowers
    dic_wr['best_names'] = best_names

    np.save(f'data_figures_2_5/dic_{weather_regime}.npy',dic_wr)

['Z500', 'U200', 'All', 'OLR', 'OLR', 'U10', 'OLR', 'U10']
['OHC700', 'OHC700', 'OHC300', 'OHC700', 'OHC700', 'OHC700', 'SSH', 'IT']
['STL\n28cm', 'SWVL\nfull', 'STL\n1m', 'SWVL\nfull', 'SD', 'SWVL\n1m', 'STL\n28cm', 'SWVL\nfull']
['All', 'All', 'OLR', 'Z500', 'Z500', 'All', 'OLR', 'U200']
['OHC300', 'OHC300', 'OHC50', 'OHC50', 'OHC100', 'OHC300', 'OHC700', 'SSH']
['STL\n28cm', 'SWVL\n7cm', 'SWVL\n7cm', 'All', 'STL\nfull', 'SWVL\n1m', 'STL\n28cm', 'SD']
['Z500', 'U200', 'All', 'All', 'All', 'U10', 'All', 'U200']
['SST', 'OHC200', 'OHC700', 'OHC200', 'OHC200', 'OHC300', 'OHC700', 'OHC300']
['STL\nfull', 'SWVL\n1m', 'STL\n1m', 'STL\n1m', 'SWVL\nfull', 'SWVL\n1m', 'SWVL\n1m', 'STL\n1m']
['U200', 'Z500', 'Z500', 'Z500', 'U200', 'All', 'U200', 'Z500']
['All', 'SSH', 'SSH', 'SSH', 'MLD', 'SSH', 'MLD', 'OHC300']
['STL\n7cm', 'STL\n7cm', 'All', 'STL\n28cm', 'STL\nfull', 'SWVL\n28cm', 'STL\nfull', 'STL\nfull']
['U200', 'OLR', 'U10', 'U200', 'OLR', 'U10', 'Z500', 'U200']
['All', 'IT', 'OHC50', '

# stratify by wr and season

In [23]:
counter = 0 
import string
alphabet = list(string.ascii_lowercase)

dic_accs = {}
dic_uppers = {}
dic_lowers = {}
dic_best_names = {}

for wr_temp in range(len(names_wr)):
    dic_accs[wr_temp] = {}
    dic_uppers[wr_temp] = {}
    dic_lowers[wr_temp] = {}
    dic_best_names[wr_temp] = {}
    
    indices_seasons = {
        "Winter": [12, 1, 2],
        "Spring": [3, 4, 5],
        "Summer": [6, 7, 8],
        "Fall": [9, 10, 11],
    }

    acc_persistence_seasons = {}
    for season in indices_seasons.keys():
        acc_persistence_temp = []

        for wk in range(1, 9):
            df_temp = df_shifts[[f"week{0}", f"week{wk}"]].dropna()
            df_temp = df_temp[(df_temp[f"week{wk}"] == wr_temp)|(df_temp[f"week{0}"] == wr_temp)]
            df_temp = df_temp[
                (df_temp.index.month == indices_seasons[season][0])
                | (df_temp.index.month == indices_seasons[season][1])
                | (df_temp.index.month == indices_seasons[season][2])
            ]
            acc_persistence_temp.append(
                accuracy_score(df_temp[f"week{0}"], df_temp[f"week{wk}"])
            )
        acc_persistence_seasons[season] = acc_persistence_temp
    
    np.save(f'data_figures_2_5/acc_persistence_seasons_{wr_temp}.npy',acc_persistence_seasons)
    
    indices_seasons = {
        "Winter": [12, 1, 2],
        "Spring": [3, 4, 5],
        "Summer": [6, 7, 8],
        "Fall": [9, 10, 11],
    }

    weeks = np.arange(1, 9)
    for season, letter in zip(indices_seasons.keys(), ["a) ", "b) ", "c) ", "d) "]):
        dic_accs[wr_temp][season] = {}
        dic_uppers[wr_temp][season] = {}
        dic_lowers[wr_temp][season] = {}
        dic_best_names[wr_temp][season] = {}
        for indices_component, component, label, color in zip(
            [indices_atmosphere, indices_ocean, indices_land],
            ["atm", "ocn", "lnd"],
            ["atmosphere", "ocean", "land"],
            ["mediumorchid", "darkblue", "orange"],
        ):
            
            all_accs = []
            all_uppers = []
            all_lowers = []
            for ivar in range(len(indices_component)):
                name_temp = list_vars[indices_component[ivar]]
                acc_temp = []
                upper_temp = []
                lower_temp = []
                for week in weeks:
                    results_temp = pd.read_csv(
                        f"{path_results}results_{name_temp}_week{week}.csv",
                        index_col=0,
                        parse_dates=True,
                    )
                    results_temp = results_temp[
                        (results_temp.index.month == indices_seasons[season][0])
                        | (results_temp.index.month == indices_seasons[season][1])
                        | (results_temp.index.month == indices_seasons[season][2])
                    ]

                    results_temp = results_temp[results_temp["y_true"] == wr_temp]
                    
                    acc_mean, acc_upper, acc_lower = compute_accuracy_and_range(results_temp)
                    acc_temp.append(acc_mean)
                    upper_temp.append(acc_upper)
                    lower_temp.append(acc_lower)
                all_accs.append(acc_temp)
                all_uppers.append(upper_temp)
                all_lowers.append(lower_temp)

            acc_temp = []
            upper_temp = []
            lower_temp = []
            for week in weeks:
                results_temp = pd.read_csv(
                    f"{path_results}results_{component}_week{week}.csv",
                    index_col=0,
                    parse_dates=True,
                )
                results_temp = results_temp[
                    (results_temp.index.month == indices_seasons[season][0])
                    | (results_temp.index.month == indices_seasons[season][1])
                    | (results_temp.index.month == indices_seasons[season][2])
                ]

                results_temp = results_temp[(results_temp["y_true"] == wr_temp)|\
                    (results_temp["y_predicted"] == wr_temp)]

                acc_mean, acc_upper, acc_lower = compute_accuracy_and_range(results_temp)
                acc_temp.append(acc_mean)
                upper_temp.append(acc_upper)
                lower_temp.append(acc_lower)
            all_accs.append(acc_temp)
            all_uppers.append(upper_temp)
            all_lowers.append(lower_temp)

            best_acc = np.max(np.array(all_accs), axis=0)
            mean_acc = np.mean(np.array(all_accs), axis=0)
            diffs = best_acc - mean_acc
            max_diff = 8
            min_diff = 0
            diffs = (diffs - min_diff) / (max_diff - min_diff)

            ind_max_acc = np.argmax(np.array(all_accs), axis=0)
            max_acc_upper = [np.array(all_uppers)[ind_max_acc[i], i] for i in range(len(ind_max_acc))]
            max_acc_lower = [np.array(all_lowers)[ind_max_acc[i], i] for i in range(len(ind_max_acc))]

            best_model = []
            for ind in ind_max_acc:
                if ind==len(all_accs)-1:
                    best_model.append('ALL')
                else:
                    best_model.append(" ".join(list_vars[indices_component[ind]].split("_")[:-1]))

            dic_accs[wr_temp][season][component] = np.max(np.array(all_accs), axis=0)
            dic_uppers[wr_temp][season][component] = max_acc_upper
            dic_lowers[wr_temp][season][component] = max_acc_lower
            dic_best_names[wr_temp][season][component] = best_model

np.save(f'data_figures_2_5/dic_accs_stratified.npy',dic_accs)
np.save(f'data_figures_2_5/dic_uppers_stratified.npy',dic_uppers)
np.save(f'data_figures_2_5/dic_lowers_stratified.npy',dic_lowers)
np.save(f'data_figures_2_5/dic_names_stratified.npy',dic_best_names)